# Kalman Macro Event V4 — 2017→Now\n\nAdds PIT-safe U.S. macro announcement surprise features to the existing Historical V3 matrices and replays the same V3 return/regime engine as a V4 candidate.\n\n**Research only:** Toss execution OFF / Neon write OFF / production promotion OFF.\n

In [ ]:
# KALMAN Macro Event V4 — one-cell Colab runner
# Research only: Toss OFF / Neon write OFF / production promotion OFF

from google.colab import drive
import json, os, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

PINNED_SHA = "8bbba91f98fa2f2b9edddea3d156bad50247d312"
SOURCE_BRANCH = "feature/macro-event-v1-20260916"
REPO = Path("/content/Codex_macro_v4")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MACRO_INPUT = DRIVE_ROOT / "Market_Macro/v1/raw/us_macro_events_normalized.parquet"
V4_TAG = "20260916_macro_event_v4_001"

def run(cmd, *, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd, env=env)

drive.mount("/content/drive", force_remount=False)

if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git", "clone", "--branch", SOURCE_BRANCH,
    "https://github.com/kimtk94/Codex.git", REPO
])
run(["git", "-C", REPO, "checkout", "--detach", PINNED_SHA])
checked = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "HEAD"], text=True
).strip()
assert checked == PINNED_SHA, (checked, PINNED_SHA)

APP = REPO / "kalman-toss-gateway"

run([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "pyarrow", "scikit-learn", "requests"
])

# Reuse an existing normalized macro file when present.
# Otherwise collect 2017->today from Trading Economics using a Colab Secret.
if not MACRO_INPUT.exists():
    try:
        from google.colab import userdata
        te_key = (userdata.get("TRADINGECONOMICS_API_KEY") or "").strip()
    except Exception:
        te_key = ""

    if not te_key:
        raise RuntimeError(
            "Macro input is absent and TRADINGECONOMICS_API_KEY is not available. "
            "Add it in Colab > Secrets, or place a normalized file at: "
            + str(MACRO_INPUT)
        )

    MACRO_INPUT.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["TRADINGECONOMICS_API_KEY"] = te_key
    run([
        sys.executable, "-m",
        "research.macro_event.collect_tradingeconomics",
        "--start", "2017-01-01",
        "--end", datetime.now(timezone.utc).date().isoformat(),
        "--output", MACRO_INPUT,
    ], cwd=APP, env=env)

run([
    sys.executable,
    APP / "scripts/colab_macro_v4.py",
    "--drive-root", DRIVE_ROOT,
    "--macro-input", "Market_Macro/v1/raw/us_macro_events_normalized.parquet",
    "--v3-candidate-tag", "20260913_return_regime_v3_001",
    "--v4-candidate-tag", V4_TAG,
    "--code-sha", PINNED_SHA,
], cwd=APP)

comparison = (
    DRIVE_ROOT / "Market_Model_V2"
    / "historical_quant_2017_v4_macro_candidate"
    / V4_TAG / "v3_vs_v4_common_window.json"
)
print("\n" + "=" * 88)
print("MACRO V4 COMPLETE")
print("=" * 88)
print("PINNED_SHA:", PINNED_SHA)
print("MACRO_INPUT:", MACRO_INPUT)
print("COMPARISON :", comparison)
print("SAFETY     : research-only / Toss OFF / Neon write OFF")
print("\nV3 vs V4:")
print(comparison.read_text(encoding="utf-8"))
